### Import

In [1]:
import os
import sys
import pickle 
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt 
from sklearn.model_selection import train_test_split

### General parameters

In [2]:
path_out    = "../Data/EHR/"
path_cohort = "../Data/Cohorts/"
race_path       = "Extraction/MIMICIII/Data/Output/"
path_timeseries = "Extraction/MIMICIII/Data/Output/"

### Split ICU-Stays to Train-Validation-Test Sets

In [3]:
column_to_read = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'Bins', 'AGE', 'ETHNICITY',
                  'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG', 'ICU_LOS_H']

df_ehr = pd.read_csv(path_out + '0h_to_24h_data.csv', low_memory=False, index_col=False, usecols= column_to_read)
df_ehr = df_ehr.drop_duplicates()
df_ehr = df_ehr.sort_values(by=['ICUSTAY_ID', 'Bins'])
df_ehr = df_ehr.reset_index(drop=True)
df_ehr.head(2)

### Check Notes Availability in First 24 Hours

In [4]:
df_note_ids = pd.read_csv(path_out + 'all_note_ids.csv', low_memory=False, index_col=False)
df_note_ids = df_note_ids[['ICUSTAY_ID', 'Bins', 'Note']]
df_note_ids.head(2)

In [5]:
df_note_ids = df_note_ids[(df_note_ids.Bins >= 0) & (df_note_ids.Bins <= 23)]
pat_has_note_24h = list(df_note_ids.ICUSTAY_ID.unique())

df_ehr['has_notes'] = 0
df_ehr.loc[df_ehr.ICUSTAY_ID.isin(pat_has_note_24h), 'has_notes'] = 1

In [7]:
df_ehr.head(2)

### Fix Age

In [8]:
df_ehr = df_ehr.drop_duplicates()
df_ehr.loc[df_ehr['AGE'] >= 95, 'AGE'] = 95
df_ehr = df_ehr[df_ehr.AGE > 16]

### Fix Race

In [9]:
with open(race_path + 'race_dictionary.pkl', 'rb') as f:
    race_dictionary = pickle.load(f)

In [10]:
general_ethnicity_mapping = {
    
    'WHITE': 'White',
    'WHITE - RUSSIAN': 'White',
    'WHITE - BRAZILIAN': 'White',
    'WHITE - OTHER EUROPEAN': 'White',
    'WHITE - EASTERN EUROPEAN': 'White',
    'PORTUGUESE': 'White',
    
    'UNABLE TO OBTAIN': 'Unknown',
    'UNKNOWN/NOT SPECIFIED': 'Unknown',
    'PATIENT DECLINED TO ANSWER': 'Unknown',
    
    'OTHER': 'Other',
    'MIDDLE EASTERN': 'Other',
    'CARIBBEAN ISLAND': 'Other',
    'MULTI RACE ETHNICITY': 'Other',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 'Other',
    
    'ASIAN': 'Asian',
    'ASIAN - THAI': 'Asian',
    'ASIAN - OTHER': 'Asian',
    'ASIAN - KOREAN': 'Asian',
    'ASIAN - CHINESE': 'Asian',
    'ASIAN - FILIPINO': 'Asian',
    'ASIAN - JAPANESE': 'Asian',
    'ASIAN - CAMBODIAN': 'Asian',
    'ASIAN - VIETNAMESE': 'Asian',
    'ASIAN - ASIAN INDIAN': 'Asian',
    
    'AMERICAN INDIAN/ALASKA NATIVE': 'Native American',
    'AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGNIZED TRIBE': 'Native American',
    
    'BLACK/AFRICAN': 'Black/African American',
    'BLACK/HAITIAN': 'Black/African American',
    'BLACK/CAPE VERDEAN': 'Black/African American',
    'BLACK/AFRICAN AMERICAN': 'Black/African American',
    
    'SOUTH AMERICAN': 'Hispanic/Latino',
    'HISPANIC OR LATINO': 'Hispanic/Latino',
    'HISPANIC/LATINO - CUBAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - MEXICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - HONDURAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - DOMINICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - COLOMBIAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - SALVADORAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - GUATEMALAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - PUERTO RICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - CENTRAL AMERICAN (OTHER)': 'Hispanic/Latino'}

In [11]:
def replace_ethnicity_with_names(df, ethnicity_dict):
    
    inv_ethnicity_dict = {v: k for k, v in ethnicity_dict.items()}
    df['ETHNICITY'] = df['ETHNICITY'].map(inv_ethnicity_dict)
    
    return df

In [12]:
def categorize_ethnicity(df, new_mapping):
    
    df['ETHNICITY'] = df['ETHNICITY'].map(new_mapping)
    
    return df

In [13]:
df_ehr = replace_ethnicity_with_names(df_ehr, race_dictionary)
df_ehr = categorize_ethnicity(df_ehr, general_ethnicity_mapping)

In [14]:
def transform_race_into_id(df):
    
    dx_type = df.ETHNICITY.unique()
    dict_dx_key = pd.factorize(dx_type)[1]
    dict_dx_val = pd.factorize(dx_type)[0]
    dictionary  = dict(zip(dict_dx_key, dict_dx_val))
    df['ETHNICITY'] = df['ETHNICITY'].map(dictionary)
    
    return df, dictionary

In [15]:
df_ehr, race_dictionary = transform_race_into_id(df_ehr)

### Create Dovosion Label

In [16]:
df_ehr['Division_Label'] = (df_ehr['ETHNICITY'].astype(int)).astype(str) + "_" + (df_ehr['HOSPITAL_EXPIRE_FLAG'].astype(int)).astype(str)

In [17]:
df_ehr.head(3)

### Take first hours of ICU of patients with more than 24 hour LoS

In [18]:
max_rows = df_ehr.groupby('ICUSTAY_ID').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [19]:
def short_long_icustays(df, observation_window):
    
    df_length_o_s = df.groupby('ICUSTAY_ID').max()[['Bins']].reset_index()
    
    df_LoS_short = df_length_o_s[df_length_o_s.Bins <  23].reset_index(drop=True)
    df_LoS_long  = df_length_o_s[df_length_o_s.Bins == 23].reset_index(drop=True)
    
    short_stays = df[df.ICUSTAY_ID.isin(df_LoS_short.ICUSTAY_ID.unique())].copy()
    short_stays = short_stays.groupby('ICUSTAY_ID').head(observation_window).reset_index(drop=True)

    long_stays = df[df.ICUSTAY_ID.isin(df_LoS_long.ICUSTAY_ID.unique())].copy()
    long_stays = long_stays.groupby('ICUSTAY_ID').head(observation_window).reset_index(drop=True)
    
    return long_stays, short_stays

In [20]:
df_long, df_short = short_long_icustays(df_ehr, observation_window)

#### Short Stays

In [21]:
icustays_shorter_24h = list(df_short.ICUSTAY_ID.unique())
len(icustays_shorter_24h)

5021

#### Stays with EHR & Note

In [22]:
all_modality_df = df_long.groupby('ICUSTAY_ID', as_index=False).first()
all_modality_df = all_modality_df[(all_modality_df.has_notes == 1)].copy()
long_icustays_with_all_modality = list(all_modality_df.ICUSTAY_ID.unique())
len(long_icustays_with_all_modality)

44103

In [23]:
all_modality_short_df = df_short.groupby('ICUSTAY_ID', as_index=False).first()
all_modality_short_df = all_modality_short_df[(all_modality_short_df.has_notes == 1)].copy()
short_icustays_with_all_modality = list(all_modality_short_df.ICUSTAY_ID.unique())
len(short_icustays_with_all_modality)

4150

#### Stays with EHR Only

In [24]:
only_ehr_df = df_long[~df_long.ICUSTAY_ID.isin(long_icustays_with_all_modality)].copy()
long_icustays_with_only_ehr_modality = list(only_ehr_df.ICUSTAY_ID.unique())
len(long_icustays_with_only_ehr_modality)

2880

In [25]:
only_ehr_short_df = df_short[~df_short.ICUSTAY_ID.isin(short_icustays_with_all_modality)].copy()
short_icustays_with_only_ehr_modality = list(only_ehr_short_df.ICUSTAY_ID.unique())
len(short_icustays_with_only_ehr_modality)

871

### Spliting Train - Validation - Test

In [31]:
# split_label = 'Division_Label'
split_label = 'HOSPITAL_EXPIRE_FLAG'

In [32]:
def split_train_test(long_stays, icustay_ids, label):
    
    long_stays = long_stays[long_stays.ICUSTAY_ID.isin(icustay_ids)].reset_index(drop=True)
    
    indexing = long_stays[['HADM_ID', label]].groupby('HADM_ID').head(1)
    train_id, test_id  = train_test_split(indexing, stratify= indexing[label], test_size= 0.20, random_state= 42)
    train_id, valid_id = train_test_split(train_id, stratify= train_id[label], test_size= 0.10, random_state= 42)
    
    index_train = train_id.HADM_ID.unique()
    index_valid = valid_id.HADM_ID.unique()
    index_test  = test_id.HADM_ID.unique()

    train_df = long_stays[long_stays['HADM_ID'].isin(index_train)].copy()
    valid_df = long_stays[long_stays['HADM_ID'].isin(index_valid)].copy()
    test_df  = long_stays[long_stays['HADM_ID'].isin(index_test)].copy()
    
    train_df[[label]] = train_df[[label]].astype(int)
    valid_df[[label]] = valid_df[[label]].astype(int)
    test_df[[label]]  = test_df[[label]].astype(int)
    
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)
    
    return train_df, valid_df, test_df

In [33]:
train_ehr_txt,  valid_ehr_txt,  test_ehr_txt  = split_train_test(df_long, long_icustays_with_all_modality, split_label)
train_ehr_only, valid_ehr_only, test_ehr_only = split_train_test(df_long, long_icustays_with_only_ehr_modality, split_label)

In [34]:
concat_train_df = [train_ehr_txt, train_ehr_only]
train_df = pd.concat(concat_train_df)

concat_valid_df = [valid_ehr_txt, valid_ehr_only]
valid_df = pd.concat(concat_valid_df)

concat_test_df  = [test_ehr_txt,  test_ehr_only]
test_df  = pd.concat(concat_test_df)

train_icustays = list(train_df.ICUSTAY_ID.unique())
valid_icustays = list(valid_df.ICUSTAY_ID.unique())
test_icustays  = list(test_df.ICUSTAY_ID.unique())

### Select ICU-Stays

In [35]:
icu_to_hosp_map = df_long.set_index('ICUSTAY_ID')['HADM_ID'].to_dict()

hosp_dataset_assignment = {}

def assign_dataset(icu_list, dataset_name):
    for icu_id in icu_list:
        hosp_id = icu_to_hosp_map[icu_id]
        if hosp_id not in hosp_dataset_assignment:
            hosp_dataset_assignment[hosp_id] = dataset_name

In [36]:
assign_dataset(train_icustays, 'train')
assign_dataset(valid_icustays, 'valid')
assign_dataset(test_icustays , 'test')

train_df_final = df_long[df_long['HADM_ID'].map(hosp_dataset_assignment) == 'train']
valid_df_final = df_long[df_long['HADM_ID'].map(hosp_dataset_assignment) == 'valid']
test_df_final  = df_long[df_long['HADM_ID'].map(hosp_dataset_assignment) == 'test']

concat_train_df= [df_short, train_df_final]
train_df_final = pd.concat(concat_train_df)

train_icustays = list(train_df_final.ICUSTAY_ID.unique())
valid_icustays = list(valid_df_final.ICUSTAY_ID.unique())
test_icustays  = list(test_df_final.ICUSTAY_ID.unique())

### Filter Text of First 24 Hours

In [37]:
note_ids_24h = list(df_note_ids.Note.unique())

### Combine Long and Short ICU-Stay Lists

In [38]:
long_icustays_with_all_modality.extend(short_icustays_with_all_modality)
long_icustays_with_only_ehr_modality.extend(short_icustays_with_only_ehr_modality)

icustays_with_all_modality = long_icustays_with_all_modality
icustays_with_only_ehr_modality = long_icustays_with_only_ehr_modality

### Save Data

In [39]:
with open(path_cohort + "icustays_ehr_text", "wb") as fp:   
    pickle.dump(icustays_with_all_modality, fp)
    
with open(path_cohort + "icustays_ehr_only", "wb") as fp:   
    pickle.dump(icustays_with_only_ehr_modality, fp)
    
with open(path_cohort + "short_icustays", "wb") as fp:   
    pickle.dump(icustays_shorter_24h, fp)
    
with open(path_cohort + "train_icustays", "wb") as fp:   
    pickle.dump(train_icustays, fp)
    
with open(path_cohort + "valid_icustays", "wb") as fp:   
    pickle.dump(valid_icustays, fp)
    
with open(path_cohort + "test_icustays", "wb") as fp:   
    pickle.dump(test_icustays, fp)
    
with open(path_cohort + "note_ids_24h", "wb") as fp:   
    pickle.dump(note_ids_24h, fp)